# Raster — JAXAの月降水量を検索して縮小表示する

```python
results = search(bbox=BBOX, time=TIME_RANGE,
                 providers=[ProviderId.JAXA_GSMAP_PREVIEW], limit=1)
resource = next(r for r in results if r.reference.parameters["asset_key"] == "PRECIP")
with resource.open("rasterio", runtime=rasterio) as dataset:
    preview = dataset.read(1, window=dataset.window(*BBOX),
                           out_shape=(64, 64), masked=True)
```

追加設定・APIキーなしで、組み込みProviderからJAXA Earth GSMaP月降水量を検索します。
入力は **BBOX** の1箇所です。既定は日本周辺（135–145E / 30–40N）。検索結果から
`PRECIP` assetを明示選択し、COG全体を展開せず64×64へ縮小読取します。

このProviderは検証済みの2021年2月・level 0・東経0〜180度だけを対象にし、
Catalog 1件＋Item 1件の最大2文書を取得します。JAXA全製品・全期間の検索ではありません。


## 出典・利用条件

本データはJAXA Earth APIを使用して取得したGSMaP月降水量です。
[公式サイトと利用条件](https://data.earth.jaxa.jp/ja/) ·
[組み込みProviderの仕様](../docs/api/adapters/stac.md)

JAXA Earth APIは現時点で登録・APIキー・回数制限を設けていません。データごとのライセンスは
STAC上の情報を確認し、公開成果にはJAXA Earth APIを利用した旨を明記してください。
APIとデータは現状有姿で、仕様変更や公開終了の可能性があります。


## 準備

組み込みJAXA Providerを含む固定コミットと、利用者側Runtimeを導入します。
Rhinestone CoreへRasterioやMatplotlibは追加しません。


In [ ]:
import os
import subprocess
import sys

if os.environ.get("RHINESTONE_SHOWCASE_LOCAL") != "1":
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "rhinestone @ git+https://github.com/u-kitazawa/rhinestone.git@114a7a10c30880dfb1655d0ce94fb2bf0f917145",
            "rasterio==1.5.2",
            "matplotlib==3.10.7",
        ]
    )
else:
    print("Local smoke: explicit checkout and caller-owned dependencies")

In [ ]:
from datetime import datetime, timezone

from rhinestone import ProviderId, search

BBOX = (135.0, 30.0, 145.0, 40.0)
TIME_RANGE = (
    datetime(2021, 2, 1, tzinfo=timezone.utc),
    datetime(2021, 2, 28, 23, 59, 59, tzinfo=timezone.utc),
)
ASSET_KEY = "PRECIP"
PREVIEW_SIZE = 64

## 固定された低解像度階層を検索

期間はProviderの固定範囲と一致させます。BBOXはItem取得後の交差判定にも使われます。
0件とProvider障害を区別し、検索成功をRaster本体の取得成功とは扱いません。


In [ ]:
results = None
results = search(
    bbox=BBOX,
    time=TIME_RANGE,
    providers=[ProviderId.JAXA_GSMAP_PREVIEW],
    limit=1,
)
for diagnostic in results.diagnostics:
    print(
        "Diagnostic:", diagnostic.source_id, diagnostic.reason, diagnostic.failure_type
    )
for candidate in results:
    print(
        candidate.title,
        "|",
        candidate.format,
        "| asset:",
        candidate.reference.parameters.get("asset_key"),
    )
    print("  ", candidate.uri)
if not results:
    if results.diagnostics:
        raise ValueError("Providerの診断を確認してください。データ不在とは限りません。")
    raise ValueError("検索0件です。BBOXと固定期間2021年2月の交差を確認してください。")

## `PRECIP` assetを明示選択

先頭結果へfallbackしません。該当assetが一意でなければ停止します。


In [ ]:
resource = None
matches = [
    candidate
    for candidate in results
    if candidate.reference.provider_id == ProviderId.JAXA_GSMAP_PREVIEW
    and candidate.reference.parameters.get("asset_key") == ASSET_KEY
]
if len(matches) != 1:
    raise ValueError(
        "PRECIP assetが検索結果に一意にありません。候補と診断を確認してください。"
    )
resource = matches[0]
print("Selected:", resource.title, resource.format, resource.uri)

## 日本周辺だけを64×64へ縮小読取

RasterioがCOGの必要範囲を読みます。TLS検証を無効化しません。環境で証明書検証に失敗する場合は
管理済みCAをGDALへ設定してください。通信量は配信・GDAL・キャッシュに依存し、64×64の配列サイズと同じではありません。


In [ ]:
import rasterio
from rasterio.enums import Resampling

from rhinestone.errors import ResourceAccessError

preview = None
try:
    with resource.open("rasterio", runtime=rasterio) as dataset:
        if str(dataset.crs) != "EPSG:4326":
            raise ValueError(f"この表示例はEPSG:4326を必要とします: {dataset.crs}")
        preview = dataset.read(
            1,
            window=dataset.window(*BBOX),
            out_shape=(PREVIEW_SIZE, PREVIEW_SIZE),
            resampling=Resampling.average,
            masked=True,
        )
        print("Dataset:", dataset.width, "x", dataset.height, "bands:", dataset.count)
except ResourceAccessError:
    print("Open失敗：配布URL、TLS/CA、Rasterio/GDALの設定を確認してください。")
    raise
if preview.count() == 0:
    raise ValueError("指定範囲に有効な降水量画素がありません。")
print(
    "Preview:",
    preview.shape,
    "valid:",
    preview.count(),
    "min/max:",
    float(preview.min()),
    float(preview.max()),
)

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(7, 6))
image = axis.imshow(
    preview,
    origin="upper",
    extent=BBOX,
    cmap="Blues",
)
axis.set(
    title="JAXA GSMaP monthly precipitation preview — February 2021",
    xlabel="Longitude",
    ylabel="Latitude",
)
figure.colorbar(image, ax=axis, label="Stored precipitation value (mm/hr)")
figure.text(0.05, 0.01, "Source: JAXA Earth API / GSMaP", fontsize=8)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## 境界

- `BBOX`は日本周辺へ変更できますが、組み込みProviderの経度範囲は東経0〜180度です。
- 期間・製品・levelの変更はこの固定プレビューでは扱いません。
- 表示値の単位・scale・offsetはSTAC metadataと提供元仕様を確認してください。この例は確認済みの
  `PRECIP`保存値を表示し、月合計などへ換算しません。
- Rasterioの縮小読取とMatplotlibの可視化はRhinestoneではなく利用者側Runtimeの責務です。
- ライブ配信先の可用性は通常CIの必須条件にしません。fixtureで検索・明示選択・縮小読取契約を検証します。
